In [ ]:
import pytorch3d
from pytorch3d.renderer import (
    MeshRasterizer,
    RasterizationSettings,
)


In [ ]:
import numpy as np
import torch
from pytorch3d.renderer import PerspectiveCameras, MeshRasterizer, RasterizationSettings
from pytorch3d.structures import Meshes
from pytorch3d.renderer.mesh.textures import TexturesVertex

# ----------------------------
# trimesh -> PyTorch3D Meshes
# ----------------------------
def trimesh_to_p3d(tri_mesh, device):
    V = torch.tensor(tri_mesh.vertices, dtype=torch.float32, device=device)
    F = torch.tensor(tri_mesh.faces, dtype=torch.int64, device=device)
    # dummy vertex colors
    C = torch.ones_like(V, device=device)
    tex = TexturesVertex(verts_features=C[None, ...])  # (1,V,3)
    return Meshes(verts=[V], faces=[F], textures=tex)

# -------------------------------------------
# camera from (cam2world, fov, width, height)
# -------------------------------------------
def p3d_camera_from_clo(cam2world_4x4, fov_deg, width, height, device, assume_opengl_cam=True):
    """
    cam2world: X_w = R_cw X_c + t_cw  (same as pyrender)
    PyTorch3D rasterizer expects points in front to have z_cam > 0.
    If your camera is OpenGL-style (looking along -Z), set assume_opengl_cam=True
    to flip into PyTorch3D's convention.
    """
    R_cw = cam2world_4x4[:3, :3]
    t_cw = cam2world_4x4[:3, 3]

    # world -> camera
    R_wc = R_cw.T
    t_wc = -R_wc @ t_cw

    if assume_opengl_cam:
        # Flip from OpenGL-like cam (-Z forward) to P3D (+Z forward)
        S = np.diag([1.0, -1.0, -1.0]).astype(np.float32)          # (3,3)
        R_wc = S @ R_wc
        t_wc = S @ t_wc

    R = torch.tensor(R_wc, dtype=torch.float32, device=device)[None, ...]
    T = torch.tensor(t_wc, dtype=torch.float32, device=device)[None, :]

    # intrinsics from vertical FOV (square pixels)
    fov_rad = np.deg2rad(fov_deg)
    fy = (height / 2.0) / np.tan(fov_rad / 2.0)
    fx = fy
    cx = (width  - 1) / 2.0
    cy = (height - 1) / 2.0

    focal_length    = torch.tensor([[fx, fy]], dtype=torch.float32, device=device)
    principal_point = torch.tensor([[cx, cy]], dtype=torch.float32, device=device)
    image_size      = torch.tensor([[height, width]], dtype=torch.float32, device=device)

    return PerspectiveCameras(
        R=R, T=T,
        focal_length=focal_length,
        principal_point=principal_point,
        image_size=image_size,
        in_ndc=False,
        device=device
    )

# ------------------------------------------
# merge many trimesh into one P3D for 1-pass
# ------------------------------------------
def merge_meshes_for_single_pass(mesh_list_trimesh, device):
    verts_all, faces_all, splits = [], [], []
    base = 0
    for tm in mesh_list_trimesh:
        V = torch.tensor(tm.vertices, dtype=torch.float32, device=device)
        F = torch.tensor(tm.faces, dtype=torch.int64, device=device) + base
        verts_all.append(V)
        faces_all.append(F)
        splits.append((base, base + V.shape[0]))
        base += V.shape[0]
    Vcat = torch.cat(verts_all, dim=0)
    Fcat = torch.cat(faces_all, dim=0)
    C = torch.ones_like(Vcat, device=device)
    tex = TexturesVertex(verts_features=C[None, ...])
    merged = Meshes(verts=[Vcat], faces=[Fcat], textures=tex)
    return merged, splits

# ---------------------------------------------------------
# visible mask per mesh (aligned with input mesh order)
# ---------------------------------------------------------
@torch.no_grad()
def visible_vertex_masks_for_meshes(mesh_list_trimesh, cameras, image_size_hw,
                                    faces_per_pixel=5, cull_backfaces=False, strict_vertex=False, bary_thresh=0.98):
    """
    Returns:
      masks: list of np.bool arrays, one per mesh in mesh_list_trimesh
    """
    device = cameras.device
    H, W = image_size_hw
    merged, splits = merge_meshes_for_single_pass(mesh_list_trimesh, device)

    rast = RasterizationSettings(
        image_size=(H, W),
        faces_per_pixel=faces_per_pixel,
        cull_backfaces=cull_backfaces
    )
    rasterizer = MeshRasterizer(cameras=cameras, raster_settings=rast)
    frags = rasterizer(meshes_world=merged)  # batch=1

    pix_to_face = frags.pix_to_face[0]  # (H,W,K)
    fg_faces = pix_to_face[pix_to_face >= 0]  # all hit faces over K
    # debug:
    # print("[debug] hit pixels:", int(fg_faces.numel()), "unique faces:", int(fg_faces.unique().numel()))

    print("-"*50)
    print("[debug] hit pixels:", int(fg_faces.numel()), "unique faces:", int(fg_faces.unique().numel()))
    fg_faces = frags.pix_to_face[0][frags.pix_to_face[0] >= 0]
    print("hit pixels:", int(fg_faces.numel()), "unique faces:", int(fg_faces.unique().numel()))
    print("="*50)

    F = merged.faces_padded()[0]  # (Ftot, 3)
    Vtot = merged.verts_padded()[0].shape[0]

    face_visible = torch.zeros(F.shape[0], dtype=torch.bool, device=device)
    if fg_faces.numel() > 0:
        face_visible[fg_faces.unique()] = True

    vert_visible_all = torch.zeros(Vtot, dtype=torch.bool, device=device)
    if face_visible.any():
        vis_faces = torch.where(face_visible)[0]
        vis_face_verts = F[vis_faces].reshape(-1)
        vert_visible_all[vis_face_verts] = True

    if strict_vertex and fg_faces.numel() > 0:
        # require a pixel where a vertex corner owns ~100% bary weight
        bary = frags.bary_coords[0]  # (H,W,K,3)
        fg = pix_to_face >= 0
        strict = torch.zeros(Vtot, dtype=torch.bool, device=device)
        for corner in (0, 1, 2):
            near = fg & (bary[..., corner] >= bary_thresh)
            f_here = pix_to_face[near]
            if f_here.numel() > 0:
                v_here = F[f_here, corner]
                strict[v_here] = True
        vert_visible_all = strict

    # split back to per-mesh masks
    masks = []
    for s0, s1 in splits:
        masks.append(vert_visible_all[s0:s1].detach().cpu().numpy())
    return masks

# --------------------------------------------
# (Optional) per-vertex pixel coords & in-bounds
# --------------------------------------------
@torch.no_grad()
def project_vertices_pixels(tri_mesh, cameras, image_size_hw, y_flip=False):
    H, W = image_size_hw
    dev = cameras.device
    V = torch.tensor(tri_mesh.vertices, dtype=torch.float32, device=dev)
    F = torch.tensor(tri_mesh.faces, dtype=torch.int64, device=dev)  # not used here, but available

    # world->screen in pixels
    V_scr = cameras.transform_points_screen(V[None, ...], image_size=(H, W))[0]  # (V,3)
    uv = V_scr[:, :2]
    if y_flip:
        uv = uv.clone()
        uv[:, 1] = (H - 1) - uv[:, 1]

    # in-front check via world->view
    V_cam = cameras.get_world_to_view_transform().transform_points(V[None, ...])[0]
    z_cam = V_cam[:, 2]
    in_front = z_cam > 0
    in_img = (uv[:, 0] >= 0) & (uv[:, 0] < (W - 1 + 1e-6)) & (uv[:, 1] >= 0) & (uv[:, 1] < (H - 1 + 1e-6))
    return uv.detach().cpu().numpy(), z_cam.detach().cpu().numpy(), (in_front & in_img).detach().cpu().numpy()

In [ ]:
# build camera with flip
cams = p3d_camera_from_clo(
    cam2world_4x4=clo_camera.cam2world,
    fov_deg=clo_camera.fov,
    width=clo_camera.width,
    height=clo_camera.height,
    device=device,
    assume_opengl_cam=True   # <-- key change
)

# sanity: check z_cam of one mesh
V = torch.tensor(body_mesh.vertices, dtype=torch.float32, device=device)
z_cam = cams.get_world_to_view_transform().transform_points(V[None])[0][:,2]
print("z_cam min/max:", float(z_cam.min()), float(z_cam.max()))  # should be mostly > 0

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

IDX = 0

view_name = scene.view_name_list[IDX]
cam_name = view_name.split("__")[0]
img = plt.imread(scene.rendered_path_list[IDX])

body_mesh = scene.body_mesh_dict[view_name]
garment_mesh_list = list(scene.garment_mesh_dict_dict[view_name].values())

clo_camera = camera_info[cam_name]

cams = p3d_camera_from_clo(
    cam2world_4x4=clo_camera.cam2world,
    fov_deg=clo_camera.fov,    # 15
    width=clo_camera.width,    # 480
    height=clo_camera.height,  # 640
    device=device
)

# order your meshes exactly as you want masks returned
mesh_list = [body_mesh] + garment_mesh_list  # or just garment_mesh_list if you want per-garment only

# per-mesh visible-vertex masks (mutual occlusion handled)
masks = visible_vertex_masks_for_meshes(
    mesh_list_trimesh=mesh_list,
    cameras=cams,
    image_size_hw=(clo_camera.height, clo_camera.width),
    faces_per_pixel=5,          # a few samples helps fill edge holes
    cull_backfaces=False,       # set True if you want strict front-face only
    strict_vertex=False         # set True if you want “vertex itself hits a pixel” via barycentric ~1.0
)

# masks[i] is a boolean array of length mesh_list[i].vertices.shape[0]
# Example: split body vs garments
mask_body = masks[0]
mask_garments = masks[1:]  # aligned with garment_mesh_list

# (optional) get 2D pixel coords for each mesh (same camera, so it aligns)
uv_list, in_img_list = [], []
for m in mesh_list:
    uv, zc, in_img = project_vertices_pixels(m, cams, (clo_camera.height, clo_camera.width), y_flip=False)
    uv_list.append(uv)
    in_img_list.append(in_img)

# final “visible in image” mask per mesh (raster visibility ∧ inside image ∧ in front)
final_masks = [vis & iim for vis, iim in zip(masks, in_img_list)]

In [ ]:
import pytorch3d
from pytorch3d.renderer import (
    MeshRasterizer,
    RasterizationSettings,
)

def get_visible_garment_vert_mask(
    body_mesh,
    garment_mesh,
    cam_info,
    width,
    height,
) :
    pass

    # meshes: PyTorch3D meshes, cameras: PerspectiveCameras (K/R/t 셋업)
    raster_settings = RasterizationSettings(image_size=(height, width), faces_per_pixel=1, cull_backfaces=False)
    rasterizer = MeshRasterizer(cameras=cameras, raster_settings=raster_settings)
    fragments = rasterizer(meshes_world=meshes)  # pix_to_face: [H,W,1], zbuf: [H,W,1], bary_coords: [H,W,1,3]

    pix_to_face = fragments.pix_to_face[..., 0].cpu().numpy()  # -1이면 배경
    visible_faces = np.unique(pix_to_face[pix_to_face >= 0])

    face_visible = np.zeros(num_faces, dtype=bool)
    face_visible[visible_faces] = True

    vertex_visible = np.zeros(num_verts, dtype=bool)
    for f in visible_faces:
        i,j,k = faces[f]                 # face의 3개 정점 인덱스
        vertex_visible[[i,j,k]] = True   # 면이 보이면 정점도 보인다고 간주 (실무적으로 충분히 정확)